# Setup

## File download

In [2]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
!pip install pyspark -qq

trip_data_path = "data/raw/yellow_tripdata_2024-01.parquet"
os.makedirs('data/raw', exist_ok=True)
if os.path.exists(trip_data_path):
    print("File already exists, skipping download.")
else:
    response = requests.get("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", stream=True)
    response.raise_for_status()
    with open(trip_data_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=4194304):  # 4MB chunks to avoid memory issues and speed up download
            f.write(chunk)
    print ("File downloaded")
import socket
print(socket.gethostname())
print(socket.gethostbyname(socket.gethostname()))

The system cannot find the path specified.


File already exists, skipping download.
Cyberpower
192.168.1.22


## Setting up PySpark

In [7]:
import pyspark
import subprocess
import time
def check_java():
    try:
        result = subprocess.run(['java', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            print("Java is installed and accessible.")
            print(result.stderr)  # Java version info is usually in stderr
        else:
            print("Java is not accessible. Please check your JAVA_HOME and PATH settings.")
            print(result.stderr)
    except FileNotFoundError:
        print("Java executable not found. Please ensure Java is installed and JAVA_HOME is set correctly.")
from pyspark.sql import SparkSession
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-17" # Update this path to your actual Java installation directory
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=true pyspark-shell"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
check_java()
print ("Environment variables set for Spark")
spark = SparkSession.builder \
    .appName("Yellow Taxi Analysis") \
    .master("local[*]") \
    .config("spark.ui.enabled", "false")\
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate() # Disable Spark UI and native library check for better performance on local machine
print("Spark started on version:", spark.version)
spark_start_time = time.perf_counter()
df = spark.read.parquet(trip_data_path); spark_end_time = time.perf_counter()
spark_time = spark_end_time - spark_start_time
print(f"Time taken for Spark to read Parquet file: {spark_time:.2f} seconds")
df.show(5)
print ("DataFrame schema:")
df.printSchema()
starting_rows = df.count()
print ("DataFrame row count:", starting_rows)
print ("DataFrame columns:", df.columns)


Java is installed and accessible.
java version "22.0.1" 2024-04-16
Java(TM) SE Runtime Environment (build 22.0.1+8-16)
Java HotSpot(TM) 64-Bit Server VM (build 22.0.1+8-16, mixed mode, sharing)

Environment variables set for Spark
Spark started on version: 4.1.1
Time taken for Spark to read Parquet file: 0.07 seconds
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+

## Comparing with Pandas

In [8]:
import pandas as pd
pandas_start_time = time.perf_counter()
p_df = pd.read_parquet(trip_data_path)
pandas_end_time = time.perf_counter()
pandas_time = pandas_end_time - pandas_start_time
print(f"Time taken to read Parquet file with Pandas: {pandas_time:.2f} seconds")
print(f"Spark was {pandas_time / spark_time:.2f} times faster than Pandas for reading the Parquet file. Pandas eagerly loaded the data into memory while Spark lazily loaded the data, which is why Spark was faster for this operation. In conclusion, Spark's lazy loading and optimized execution engine make it more efficient for handling large datasets compared to Pandas, which loads the entire dataset into memory at once. Pandas is trash for big data, Spark is the way to go.")

Time taken to read Parquet file with Pandas: 0.18 seconds
Spark was 2.56 times faster than Pandas for reading the Parquet file. Pandas eagerly loaded the data into memory while Spark lazily loaded the data, which is why Spark was faster for this operation. In conclusion, Spark's lazy loading and optimized execution engine make it more efficient for handling large datasets compared to Pandas, which loads the entire dataset into memory at once. Pandas is trash for big data, Spark is the way to go.


# Data Cleaning & Feature Engineering in Spark

## Clean the data

In [9]:
# Cleanup Spark dataframes.
# Removing rows with null values in critical columns (pickup/dropoff times, locations, fare,distance)
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",  "fare_amount", "trip_distance", "PULocationID", "DOLocationID"])
print (f"{starting_rows - df.count()} rows removed due to null values. Remaining rows: {df.count()}")
starting_rows = df.count()
# Removing rows with non-positive fare amounts or trip distances
df = df.filter((df.fare_amount > 0) & (df.trip_distance > 0))
print (f"{starting_rows - df.count()} rows removed due to non-positive fare amounts or trip distances. Remaining rows: {df.count()}")
starting_rows = df.count()
starting_rows = df.count()
# Removing rows with pickup times after dropoff times
df = df.filter(df.tpep_pickup_datetime < df.tpep_dropoff_datetime)
print (f"{starting_rows - df.count()} rows removed due to pickup times after dropoff times. Remaining rows: {df.count()}")
starting_rows = df.count()
# Remove rows with ludicrous fares (e.g., fare_amount > $500)
df = df.filter(df.fare_amount <= 500)
print (f"{starting_rows - df.count()} rows removed due to ludicrous fares. Remaining rows: {df.count()}")


0 rows removed due to null values. Remaining rows: 2964624
94910 rows removed due to non-positive fare amounts or trip distances. Remaining rows: 2869714
112 rows removed due to pickup times after dropoff times. Remaining rows: 2869602
30 rows removed due to ludicrous fares. Remaining rows: 2869572


## Derive columns

In [10]:
# We will now derive new columns for analysis. We will derive trip duration in minutes, speed in miles per hour, pickup hour of day, and pickup day of week. Tip percentage will also be derived for later analysis.
from pyspark.sql import functions as F

df = df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    )
    / 60,
)
df = df.withColumn(
    "speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
)
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
df = df.withColumn("pickup_dayOfWeek", F.dayofweek("tpep_pickup_datetime"))
df = df.withColumn("tip_percentage", (F.col("tip_amount") / F.col("fare_amount")) * 100)
print(
    "Derived new columns: trip_duration_minutes, speed_mph, pickup_hour, pickup_dayOfWeek, tip_percentage"
)
df.show(5)

Derived new columns: trip_duration_minutes, speed_mph, pickup_hour, pickup_dayOfWeek, tip_percentage
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+-----------+----------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|trip_duration_minutes|         speed_mph|pickup_hour|pickup_dayOfWeek|    tip_percentage|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+-

## Spark SQL Analytics

In [21]:
# Register the DataFrame as a temporary view for Spark SQL queries
df.createOrReplaceTempView("trips")
queries = {}
# What are the top 10 busiest pickup hours, and what is the average fare and tippercentage for each?
queries[
    "top_pickup_hours"
] = """
SELECT pickup_hour, COUNT(*) AS trip_count, ROUND(AVG(fare_amount), 2) AS avg_fare, ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

# Which day of the week has the highest average trip speed? Include averagedistance and duration
queries[
    "fastest_day_of_week"
] = """
SELECT pickup_dayOfWeek, ROUND(AVG(speed_mph), 2) AS avg_speed, ROUND(AVG(trip_distance), 2) AS avg_distance, ROUND(AVG(trip_duration_minutes), 2) AS avg_duration
FROM trips
GROUP BY pickup_dayOfWeek
ORDER BY avg_speed DESC
LIMIT 1
"""

# Using a window function, rank the top 5 pickup locations by total revenue foreach day of the week.
queries[
    "top_pickup_locations_by_revenue"
] = """
SELECT *
FROM (
    SELECT 
        pickup_dayOfWeek,
        PULocationID,
        ROUND(SUM(fare_amount), 2) AS total_revenue,
        RANK() OVER (
            PARTITION BY pickup_dayOfWeek 
            ORDER BY SUM(fare_amount) DESC
        ) AS revenue_rank
    FROM trips
    GROUP BY pickup_dayOfWeek, PULocationID
)
WHERE revenue_rank <= 5
ORDER BY pickup_dayOfWeek, revenue_rank;
"""

# Calculate the cumulative trip count by hour of day (running total from hour 0 to23). At what hour does the cumulative count surpass 50% of daily trips?
queries[
    "cumulative_trip_count_by_hour"
] = """
WITH hourly_counts AS (
    SELECT pickup_hour, COUNT(*) AS trip_count
    FROM trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT pickup_hour, ROUND(SUM(trip_count) OVER (ORDER BY pickup_hour), 2) AS cumulative_trip_count
    FROM hourly_counts
)
SELECT pickup_hour
FROM cumulative_counts
WHERE cumulative_trip_count > (SELECT SUM(trip_count) FROM hourly_counts) * 0.5
ORDER BY pickup_hour
LIMIT 1
"""

# Compare average fare, distance, and tip percentage between short trips (<2miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has thehighest tip percentage?
queries[
    "trip_length_comparison"
] = """
WITH categorized AS (
    SELECT *,
        CASE
            WHEN trip_distance < 2 THEN 'Short'
            WHEN trip_distance BETWEEN 2 AND 10 THEN 'Medium'
            ELSE 'Long'
        END AS trip_length_category
    FROM trips
)
SELECT
    trip_length_category,
    AVG(fare_amount) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM categorized
GROUP BY trip_length_category
ORDER BY avg_tip_percentage DESC
"""

# Execute the queries and print results
for query_name, query in queries.items():
    print(f"\nResults for query: {query_name}")
    result_df = spark.sql(query)
    result_df.show()


Results for query: top_pickup_hours
+-----------+----------+--------+------------------+
|pickup_hour|trip_count|avg_fare|avg_tip_percentage|
+-----------+----------+--------+------------------+
|         18|    206256|   17.02|             22.78|
|         17|    200280|   18.12|             22.34|
|         16|    184947|   19.46|             21.84|
|         15|    183971|   19.11|              19.8|
|         19|    178785|   17.63|             22.86|
|         14|    178002|   19.27|              19.8|
|         13|    165326|   18.42|             19.79|
|         12|    159884|    17.8|             19.74|
|         21|    155885|    18.3|             21.88|
|         20|    155539|   18.05|             22.17|
+-----------+----------+--------+------------------+


Results for query: fastest_day_of_week
+----------------+---------+------------+------------+
|pickup_dayOfWeek|avg_speed|avg_distance|avg_duration|
+----------------+---------+------------+------------+
|              

## Performance Optimization

In [ ]:
# We want to compare time taken on uncached queries vs cached queries. We will run the top pickup hours query uncached and then cached to compare times.

# Uncached query
uncached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
uncached_end_time = time.perf_counter()
uncached_time = uncached_end_time - uncached_start_time

# Cache the DataFrame and run the same query again
df.cache()
cached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
cached_end_time = time.perf_counter()
cached_time = cached_end_time - cached_start_time
print(f"Time taken for uncached query: {uncached_time:.2f} seconds")
print(f"Time taken for cached query: {cached_time:.2f} seconds")
print(f"Caching the DataFrame resulted in a speedup of {uncached_time / cached_time:.2f} times for the top pickup hours query. Due to the way Spark optimizes execution, the first run of the query will be slower as it reads from disk and processes the data. Once cached, subsequent runs of the same query will be much faster as Spark can read from memory instead of disk, demonstrating the performance benefits of caching in Spark for repeated queries on the same dataset. But this data is small, so I see a neglible difference. Should be larger on bigger datasets")

# Write the cleaned data back as partioned
df.write.mode("overwrite").partitionBy("pickup_hour").parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet")
print("Cleaned data written back to disk, partitioned by pickup_hour. We will now read hour 17 data to see how much faster it is to read a partitioned file vs the original unpartitioned file.")
# Read the partitioned data for hour 17
partitioned_start_time = time.perf_counter()
hour_17_df = spark.read.parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet/pickup_hour=17")
partitioned_end_time = time.perf_counter()
partitioned_time = partitioned_end_time - partitioned_start_time
print(f"Time taken to read partitioned data for hour 17: {partitioned_time:.2f} seconds")
print(f"Time taken to read the original unpartitioned Parquet file: {spark_time:.2f} seconds")
print(f"Reading the partitioned data for hour 17 was {spark_time / partitioned_time:.2f} times faster than reading the original unpartitioned Parquet file. This demonstrates the performance benefits of partitioning in Spark, as it allows Spark to read only the relevant subset of data (hour 17) instead of scanning the entire dataset, resulting in significantly faster read times for queries that filter on the partition column.")

# RAG Pipeline over Transportation Documents

## Read the documents into memory

In [ ]:
import PyPDF2
files_memory = {}
for file in os.listdir("docs/"):
    files_memory[file] = PyPDF2.PdfReader(os.path.join("docs/", file))
print ("Files read:",files_memory.keys())
for path in files_memory.values():
    for page in path.pages:
        print(f'\033[38;2;199;61;144m{file}\033[0m')
        print (page.extract_text())
        print ("----- End of page -----")

## Chunk the documents

Defining a reusable function

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(files_memory, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    all_docs = []

    for filename, pdf in files_memory.items():
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text:
                continue
            
            chunks = splitter.create_documents(
                [text],
                metadatas=[{
                    "source": filename,
                    "page": i
                }]
            )
            all_docs.extend(chunks)

    return all_docs


In [ ]:
docs_1000 = chunk_documents(files_memory, 1000, 200)

print("Total chunks:", len(docs_1000))
import matplotlib.pyplot as plt

chunk_lengths = [len(doc.page_content) for doc in docs_1000]

plt.hist(chunk_lengths, bins=20)
plt.title("Chunk Size Distribution (1000)")
plt.xlabel("Chunk length (characters)")
plt.ylabel("Frequency")
plt.show()

Generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_docs(docs):
    texts = [doc.page_content for doc in docs]
    embeddings = model.encode(texts)
    return embeddings

Store in ChromaDB

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="pdf_chunks_1000")

def store_in_chroma(docs, embeddings, collection):
    collection.add(
        documents=[doc.page_content for doc in docs],
        embeddings=embeddings.tolist(),
        metadatas=[doc.metadata for doc in docs],
        ids=[str(i) for i in range(len(docs))]
    )

embeddings_1000 = embed_docs(docs_1000)
store_in_chroma(docs_1000, embeddings_1000, collection)

Testing different chunk sizes

In [ ]:
docs_500 = chunk_documents(files_memory, 500, 100)
docs_1000 = chunk_documents(files_memory, 1000, 200)
docs_2000 = chunk_documents(files_memory, 2000, 400)
emb_500 = embed_docs(docs_500)
emb_1000 = embed_docs(docs_1000)
emb_2000 = embed_docs(docs_2000)

col_500 = client.create_collection("chunks_500")
col_1000 = client.create_collection("chunks_1000")
col_2000 = client.create_collection("chunks_2000")

store_in_chroma(docs_500, emb_500, col_500)
store_in_chroma(docs_1000, emb_1000, col_1000)
store_in_chroma(docs_2000, emb_2000, col_2000)

def query_collection(collection, query, model):
    query_embedding = model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )
    
    return results["documents"]